In [2]:
import yfinance as yf

# Fetch historical data
ticker = "AAPL"
data = yf.download(ticker, period="2y", interval="1d")

# Look at the first few rows
print(data.head())

[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2024-04-01  168.496521  169.705519  167.951478  169.646063  46240500
2024-04-02  167.317276  167.812767  166.712777  167.555117  49329500
2024-04-03  168.119965  169.140674  167.059623  167.267720  47691700
2024-04-04  167.297455  170.369488  167.297455  168.754183  53704400
2024-04-05  168.050598  168.853291  167.426275  168.060503  42104800


In [3]:
# Create technical indicators
data['Returns'] = data['Close'].pct_change()
data['MA10'] = data['Close'].rolling(window=10).mean()
data['MA50'] = data['Close'].rolling(window=50).mean()

# Define what we want to predict (Did price go UP tomorrow?)
data['Target'] = (data['Close'].shift(-1) > data['Close']).astype(int)
data.dropna(inplace=True) # Clean up empty rows

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Define inputs (X) and what we want to predict (y)
X = data[['Close', 'Volume', 'MA10', 'MA50', 'Returns']]
y = data['Target']

# 2. Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and train the Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Check the score
predictions = model.predict(X_test)
print(f"Model Training Accuracy: {accuracy_score(y_test, predictions)*100:.2f}%")

Model Training Accuracy: 51.65%


In [5]:
import joblib

# 1. Save the "Brain" (The trained model)
joblib.dump(model, 'stock_predictor_model.pkl')

# 2. Save the "Recipe" (The list of columns/features)
# This ensures the app uses the exact same data format as the training
joblib.dump(X.columns.tolist(), 'model_features.pkl')

print("Success! Both files have been saved to your folder.")

Success! Both files have been saved to your folder.
